# HeartMuLa｜Kaggle 雙 GPU 音樂生成

**環境：** Kaggle 選擇 GPU T4 ×2，開啟 Internet。

**操作：** 首次依序執行所有程式格；之後只需執行「4. 生成設定」、「5. 生成音樂」與「6. 試聽與下載」。檔名可於第 4 節修改，重名時自動加上編號。


## 1. 安裝與 GPU 檢查

In [ ]:
# Import environment, installation, filesystem, and CUDA utilities.
# 匯入環境、安裝、檔案系統與 CUDA 工具。
import os
# Run installation commands in a separate process.
# 使用獨立程序執行套件安裝指令。
import subprocess
# Access the current Python interpreter.
# 取得目前的 Python 執行環境。
import sys
# Build filesystem paths safely.
# 使用 Path 管理檔案路徑。
from pathlib import Path
# Check and use CUDA GPU devices.
# 檢查並使用 CUDA GPU 裝置。
import torch

# Stop before model downloads unless exactly two CUDA GPUs are available.
# 若沒有兩張 CUDA GPU，就在下載模型前停止。
assert torch.cuda.device_count() == 2, '請在 Kaggle Settings 選擇 GPU T4 ×2。'

# Keep source code and final MP3 in Kaggle's working directory.
# 將原始碼與最終 MP3 放在 Kaggle 工作目錄。
ROOT = Path('/kaggle/working')
# Locate the official source repository.
# 設定官方原始碼路徑。
REPO = ROOT / 'heartlib'

# Store large model weights outside the Kaggle output directory.
# 將大型模型權重放在 Kaggle 輸出目錄以外。
CKPT = Path('/tmp/heartmula/ckpt')
# Create model-storage directories when needed.
# 必要時建立模型儲存目錄。
CKPT.mkdir(parents=True, exist_ok=True)

# Keep the Hugging Face cache on temporary storage.
# 將 Hugging Face 快取放到暫存空間。
os.environ['HF_HOME'] = '/tmp/heartmula/huggingface'
# Move the Xet download cache to temporary storage.
# 將 Xet 下載快取放到暫存空間。
os.environ['HF_XET_CACHE'] = '/tmp/heartmula/huggingface/xet'
# Disable tokenizer parallelism warnings.
# 關閉 tokenizer 平行處理警告。
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Download the official source if it is not already available.
# 只有尚未存在官方原始碼時才下載。
if not (REPO / 'pyproject.toml').exists():
    # Clone the upstream repository quietly.
    # 以安靜模式複製官方程式庫。
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HeartMuLa/heartlib.git', str(REPO)], check=True)

# Install compatible model dependencies without replacing PyTorch.
# 安裝相容的模型套件，避免覆蓋 Kaggle 原有 PyTorch。
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.0', 'tokenizers==0.22.1',
    'huggingface-hub==0.36.2', 'vector-quantize-pytorch==1.27.15', 'soundfile',
], check=True)

# Keep Kaggle's PyTorch and TorchAO when installing TorchTune.
# 安裝 TorchTune 時保留 Kaggle 的 PyTorch 與 TorchAO。
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchtune==0.4.0'], check=True)

# Register the official HeartMuLa package without reinstalling its dependencies.
# 註冊官方 HeartMuLa 套件，不重裝相依套件。
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO)], check=True)

# Make the actual src package take precedence over the same-named checkout folder.
# 讓真正的 src 套件優先於同名的 Git 專案目錄，避免 namespace package 衝突。
sys.path.insert(0, str(REPO / 'src'))

# Give installation checks the same import priority in their subprocesses.
# 讓安裝檢查子程序也優先載入正式套件，而非外層 Git 目錄。
os.environ['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')

# Check the critical imports before downloading large model files.
# 下載大型權重前先測試必要套件是否能匯入。
subprocess.run([
    sys.executable, '-c',
    'import torchao, torchtune, torchaudio; from heartlib import HeartMuLaGenPipeline',
], check=True)

# Display both physical GPU names.
# 顯示兩張實體 GPU 的名稱。
for gpu in range(2):
    # Report the CUDA device assigned to each stage.
    # 顯示各階段對應的 CUDA 裝置。
    print(f'GPU {gpu}: {torch.cuda.get_device_name(gpu)}')


## 2. 官方模型與 RoPE 相容修正

In [ ]:
# Import the official Hugging Face model snapshot downloader.
# 匯入 Hugging Face 官方模型下載工具。
from huggingface_hub import snapshot_download

# Download the generation metadata and both original-weight models.
# 下載生成設定與兩個原始權重模型。
for model_id, directory, patterns in [
    ('HeartMuLa/HeartMuLaGen', CKPT, ['tokenizer.json', 'gen_config.json']),
    ('HeartMuLa/HeartMuLa-oss-3B-happy-new-year', CKPT / 'HeartMuLa-oss-3B', ['config.json', '*.safetensors', '*.safetensors.index.json']),
    ('HeartMuLa/HeartCodec-oss-20260123', CKPT / 'HeartCodec-oss', ['config.json', '*.safetensors', '*.safetensors.index.json']),
]:
    # Download only the files required for inference.
    # 只下載推論所需的檔案。
    snapshot_download(repo_id=model_id, local_dir=str(directory), allow_patterns=patterns, max_workers=2)

# Locate the upstream cache-initialization implementation.
# 找到官方模型快取初始化的原始碼。
model_file = REPO / 'src/heartlib/heartmula/modeling_heartmula.py'

# Read the source before applying the known RoPE compatibility patch.
# 套用已知 RoPE 相容修正前先讀取原始碼。
source = model_file.read_text(encoding='utf-8')

# Identify the upstream cache initialization that needs adjustment.
# 定位需要調整的官方快取初始化區段。
original = '        with device:\n            self.backbone.setup_caches(max_batch_size, dtype)'

# Rebuild missing RoPE caches on the target CUDA device before setup.
# 先在指定的 CUDA 裝置重建缺少的 RoPE 快取。
replacement = (
    "        with device:\n"
    "            # Restore RoPE caches on the actual compute device.\n"
    "            # 在實際運算裝置上重建 RoPE 快取。\n"
    "            for module in self.modules():\n"
    "                # Repair position embeddings only when their cache is missing.\n"
    "                # 只修復尚未建立快取的位置 embedding。\n"
    "                if hasattr(module, 'rope_init') and not getattr(module, 'is_cache_built', True):\n"
    "                    # Initialize the missing positional cache.\n"
    "                    # 初始化缺少的位置快取。\n"
    "                    module.rope_init()\n"
    "                    # Keep the positional module on the model GPU.\n"
    "                    # 確保位置模組維持在模型所使用的 GPU。\n"
    "                    module.to(device)\n"
    "            self.backbone.setup_caches(max_batch_size, dtype)"
)

# Avoid applying the compatibility fix twice.
# 避免重複套用相容修正。
if 'module.rope_init()' not in source:
    # Refuse to patch an unrecognized upstream source layout.
    # 若官方原始碼結構改變，避免直接修改未知區段。
    assert original in source, 'HeartMuLa 的 setup_caches 已變更，請檢查官方版本。'
    # Apply the patch exactly once without touching checkpoint weights.
    # 只修正一處程式碼，不更動模型權重。
    model_file.write_text(source.replace(original, replacement, 1), encoding='utf-8')


## 3. 載入模型（僅首次）

In [ ]:
# Import standard utilities for Python package resolution.
# 匯入 Python 套件路徑所需的工具。
import sys
# Resolve filesystem paths for the package and output directory.
# 使用 Path 確認套件與輸出目錄的路徑。
from pathlib import Path
# Inspect the model's actual CUDA placement.
# 檢查模型實際使用的 CUDA 裝置。
import torch

# Prioritize the official Python package under the repository's src directory.
# 優先載入官方專案 src 目錄內的 Python 套件。
sys.path.insert(0, str(REPO / 'src'))

# Inspect any previously cached namespace-only package.
# 檢查先前可能暫存的錯誤 namespace package。
cached_heartlib = sys.modules.get('heartlib')

# Remove a namespace-only import without discarding a valid package.
# 僅移除錯誤的 namespace 快取，保留已正確匯入的套件。
if cached_heartlib is not None and getattr(cached_heartlib, '__file__', None) is None:
    # Allow Python to resolve the actual package from src.
    # 讓 Python 從 src 重新解析真正的套件。
    del sys.modules['heartlib']

# Load the official music-generation pipeline class.
# 匯入官方音樂生成 Pipeline 類別。
from heartlib import HeartMuLaGenPipeline
# Verify the source directory of the loaded package.
# 確認實際載入的是官方專案中的套件。
import heartlib

# Refuse a different module with the same package name.
# 若匯入了其他同名套件則立即停止。
assert Path(heartlib.__file__).resolve().is_relative_to(REPO / 'src'), f'heartlib 匯入來源錯誤：{heartlib.__file__}'

# Assign token generation and audio decoding to separate GPUs.
# 將 token 生成與音訊解碼分配到不同 GPU。
devices = {'mula': torch.device('cuda:0'), 'codec': torch.device('cuda:1')}

# Reuse an existing pipeline to avoid duplicating large models in VRAM.
# 重複執行此格時沿用既有 Pipeline，避免 VRAM 中出現第二份模型。
if globals().get('pipe') is None:
    # Load both model components once without lazy loading.
    # 僅載入一次雙 GPU 模型，停用 Lazy Loading。
    pipe = HeartMuLaGenPipeline.from_pretrained(
        str(CKPT), device=devices,
        dtype={'mula': torch.float16, 'codec': torch.float32},
        version='3B', lazy_load=False,
    )
    # Confirm the initial model loading has completed.
    # 確認首次載入模型已完成。
    print('已首次載入模型；後續生成請執行第 4、5、6 節。')
else:
    # Confirm that the existing pipeline is the intended implementation.
    # 確認既有 Pipeline 是正確的官方類別。
    assert isinstance(pipe, HeartMuLaGenPipeline), '既有 pipe 不是 HeartMuLaGenPipeline。'
    # Report that the existing model will be reused.
    # 顯示正在沿用已載入的模型。
    print('沿用既有模型，未再次載入權重。')

# Require both model components to remain loaded instead of lazy loading.
# 確認兩個模型均已載入，沒有使用 Lazy Loading。
assert not pipe.lazy_load and pipe._mula is not None and pipe._codec is not None, '雙 GPU 模型未完整載入。'

# Verify that all HeartMuLa parameters are on GPU 0.
# 確認 HeartMuLa 所有參數均位於 GPU 0。
assert {p.device for p in pipe.mula.parameters()} == {devices['mula']}, 'HeartMuLa 並未完整位於 GPU 0。'

# Verify that all HeartCodec parameters are on GPU 1.
# 確認 HeartCodec 所有參數均位於 GPU 1。
assert {p.device for p in pipe.codec.parameters()} == {devices['codec']}, 'HeartCodec 並未完整位於 GPU 1。'

# Show allocated memory for both CUDA devices.
# 顯示兩張 GPU 已配置的記憶體用量。
for gpu in range(2):
    # Report memory still occupied by loaded model parameters and caches.
    # 回報模型參數與快取目前佔用的記憶體。
    print(f'GPU {gpu}：{torch.cuda.memory_allocated(gpu) / 1024**3:.2f} GiB 已配置')


## 4. 生成設定

In [ ]:
# Define the lyrics for the next generation.
# 設定下一首音樂使用的歌詞。
LYRICS = """
[Verse]
十年生死兩茫茫
不思量，卻自難忘
千里孤墳隔月光
無處話淒涼

[Chorus]
夜來幽夢，忽然還鄉
小軒窗前，你正梳妝
相顧無言，淚落千行
一醒只剩，滿地月光

[Bridge]
料得年年最斷腸
明月夜，短松岡

[Chorus]
隔著十年，隔著陰陽
明月仍照，那座松岡
""".strip()

# Describe the musical style using comma-separated tags.
# 使用逗號分隔的標籤描述曲風。
TAGS = 'soft,sad,ballad,longing'

# Limit the next audio generation to this many seconds.
# 設定下一首音訊的生成秒數上限。
DURATION = 20

# Set the MP3 filename in the same configuration cell.
# 在同一個設定格指定 MP3 檔名。
OUTPUT_NAME = '十年生死兩茫茫.mp3'


## 5. 生成音樂（可重複執行）

In [ ]:
# Import timing utilities to report generation duration.
# 匯入計時工具以顯示音樂生成耗時。
import time

# Access the official pipeline module to report frame progress in this cell.
# 取得官方 Pipeline 模組，讓逐幀進度顯示在本格下方。
import heartlib.pipelines.music_generation as music_generation

# Save the latest lyric text to a file accepted by the official pipeline.
# 將最新歌詞寫入官方 Pipeline 可讀取的文字檔。
lyrics_file = ROOT / 'heartmula_lyrics.txt'
# Update the input file on every new generation.
# 每次生成時更新歌詞檔，確保使用最新內容。
lyrics_file.write_text(LYRICS, encoding='utf-8')

# Save the latest style description to its input file.
# 將最新曲風標籤寫入輸入檔案。
tags_file = ROOT / 'heartmula_tags.txt'
# Update style tags before each generation.
# 每次生成之前更新曲風標籤。
tags_file.write_text(TAGS, encoding='utf-8')

# Interpret the filename configured above as a local MP3 filename.
# 將設定格指定的檔名視為本地 MP3 檔名。
filename = Path(OUTPUT_NAME)

# Reject directory paths and non-MP3 extensions.
# 防止輸出到其他資料夾，並限制副檔名為 MP3。
assert filename.name == OUTPUT_NAME and filename.suffix.lower() == '.mp3', 'OUTPUT_NAME 必須是單純的 .mp3 檔名。'

# Use the configured filename for the first generation.
# 第一次生成直接使用指定的檔名。
output = ROOT / filename

# Start numbering at 002 only when the target file already exists.
# 只有指定檔案已存在時，才從 002 開始加上流水號。
index = 2

# Preserve previous music by adding a suffix to repeated filenames.
# 同名檔案已存在時自動加上編號，避免覆蓋先前的音樂。
while output.exists():
    # Choose the next available numbered MP3 filename.
    # 依序尋找下一個尚未使用的 MP3 檔名。
    output = ROOT / f'{filename.stem}_{index:03d}.mp3'
    # Move to the next number if another file has the same name.
    # 若該編號也已使用，繼續尋找下一個編號。
    index += 1

# Keep the upstream progress implementation for restoration after this run.
# 暫存官方進度機制，於本次執行結束後還原。
original_tqdm = music_generation.tqdm

# Print actual generated-frame progress, even without Notebook widgets.
# 即使 Notebook 沒有啟用 widget，也能顯示實際逐幀生成進度。
def show_progress(frame_indices):
    # Read the total number of 80 ms frames allowed in this generation.
    # 取得本次允許生成的 80 毫秒音訊幀數量。
    total = len(frame_indices)
    # Report roughly every 10% to avoid excessive log lines.
    # 大約每 10% 回報一次，避免輸出過多紀錄。
    interval = max(1, total // 10)
    # Indicate that frame-by-frame generation has begun.
    # 顯示已開始逐幀產生音訊。
    print(f'音訊生成中：最多 {total} 幀（{DURATION} 秒）', flush=True)
    # Pass each original frame index to the official generation loop.
    # 將每個原始幀索引交回官方的生成迴圈。
    for frame in frame_indices:
        # Report completed frames at regular intervals.
        # 定期回報已完成的音訊幀數量。
        if frame > 0 and frame % interval == 0:
            # Show the generated-duration limit fraction and wall-clock time.
            # 顯示生成長度上限的進度，以及實際經過時間。
            print(f'長度進度 {frame / total:.0%}｜約 {frame * 0.08:.1f}/{DURATION} 秒｜耗時 {time.perf_counter() - start:.0f} 秒', flush=True)
        # Keep inference behavior unchanged.
        # 保持官方推論行為不變。
        yield frame
    # Report 100% only if the full configured frame limit is reached.
    # 僅在完整達到設定的音訊幀數上限時顯示 100%。
    print('長度進度 100%｜已達音訊長度上限', flush=True)

# Record which CUDA devices actually execute the selected model layers.
# 記錄本次實際執行指定模型層的 CUDA 裝置。
used_gpus = set()

# Capture the device of each selected layer's output tensor.
# 以指定模型層的輸出 Tensor 裝置確認實際運算位置。
def record_gpu(module, inputs, result):
    # Identify the device used by this model operation.
    # 取得本次模型運算實際使用的 GPU 編號。
    gpu = result.device.index
    # Announce audio decoding when the GPU 1 decoder first runs.
    # 當 GPU 1 的解碼器首次執行時顯示解碼狀態。
    if gpu == 1 and gpu not in used_gpus:
        # Stream the decoding-stage message into this cell.
        # 在本格輸出區即時顯示音訊解碼訊息。
        print('[2/3] GPU 1 正在解碼音訊...', flush=True)
    # Accumulate GPU indices observed during this generation.
    # 記錄本次生成過程實際執行運算的 GPU 編號。
    used_gpus.add(gpu)

# Observe token-generation computations on GPU 0.
# 監測 GPU 0 上的 token 生成運算。
mula_hook = pipe.mula.codebook0_head.register_forward_hook(record_gpu)

# Observe waveform-decoder computations on GPU 1.
# 監測 GPU 1 上的音訊解碼運算。
codec_hook = pipe.codec.decoder.scalar_model.decoder[0].register_forward_hook(record_gpu)

# Start timing the current generation run.
# 開始計算本次音樂生成所花費的時間。
start = time.perf_counter()

# Report the new output name before time-consuming model inference.
# 在耗時的模型推論前，先顯示本次音樂輸出檔名。
print(f'[1/3] 正在生成 {output.name}，音訊長度上限 {DURATION} 秒...', flush=True)

# Use text progress for this run only, without modifying model weights.
# 只在本次執行使用文字進度顯示，不修改模型權重。
music_generation.tqdm = show_progress

# Ensure monitoring hooks and official progress are restored on failure.
# 即使推論失敗，也要還原官方進度機制並移除監測 hook。
try:
    # Disable gradients to avoid unnecessary inference memory use.
    # 停用梯度計算，避免推論時產生不必要的記憶體用量。
    with torch.no_grad():
        # Reuse both resident models through the official pipeline's generation API.
        # 透過官方 Pipeline 重用已駐留的雙 GPU 模型完成生成與解碼。
        pipe(
            {'lyrics': str(lyrics_file), 'tags': str(tags_file)},
            save_path=str(output),
            max_audio_length_ms=DURATION * 1000,
            temperature=1.0, topk=50, cfg_scale=1.5,
        )
finally:
    # Restore the official progress function for other pipeline uses.
    # 還原官方進度函式，不影響其他 Pipeline 使用情境。
    music_generation.tqdm = original_tqdm
    # Remove the GPU 0 monitoring hook after this run.
    # 本次執行結束後移除 GPU 0 的監測 hook。
    mula_hook.remove()
    # Remove the GPU 1 monitoring hook after this run.
    # 本次執行結束後移除 GPU 1 的監測 hook。
    codec_hook.remove()

# Confirm that generation and MP3 saving returned from the pipeline.
# 確認 Pipeline 已完成生成與 MP3 儲存流程。
print('[3/3] 已完成音訊生成與儲存，正在檢查結果...', flush=True)

# Verify that the models actually performed operations on both GPUs.
# 確認兩張 GPU 都實際執行了模型運算。
assert used_gpus == {0, 1}, f'雙 GPU 運算驗證失敗：{used_gpus}'

# Require the generated MP3 to exist and contain nonempty audio data.
# 確認本次成功產生非空的 MP3 音訊檔案。
assert output.is_file() and output.stat().st_size > 1024, '找不到有效的 MP3。'

# Report actual memory allocations on both GPUs after generation.
# 回報本次生成後兩張 GPU 的實際記憶體配置。
for gpu in range(2):
    # Show that the model remains resident for the next generation.
    # 顯示模型仍駐留在 GPU，供下一次生成直接使用。
    print(f'GPU {gpu}：已驗證運算；VRAM {torch.cuda.memory_allocated(gpu) / 1024**3:.2f} GiB')

# Report this run's generated file and elapsed minutes.
# 顯示本次生成的檔案名稱與實際耗時。
print(f'MP3：{output}；生成耗時：{(time.perf_counter() - start) / 60:.2f} 分鐘')


## 6. 試聽與下載


In [ ]:
# Import notebook audio playback and download links.
# 匯入 Notebook 音訊播放與下載連結工具。
from IPython.display import Audio, FileLink, display

# Preview the completed MP3 in Kaggle.
# 在 Kaggle 中試聽完成的 MP3。
display(Audio(filename=str(output)))

# Provide a download link for the generated MP3.
# 提供生成 MP3 的下載連結。
display(FileLink(str(output)))
